In [25]:
import pandas as pd
import numpy as np

In [26]:
# Load the three datasets
orders = pd.read_csv('olist_orders_dataset.csv')
items = pd.read_csv('olist_order_items_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')

In [27]:
# Merge items and orders on 'order_id'
df_merged = pd.merge(
    items,
    orders[['order_id', 'order_status', 'order_purchase_timestamp']],
    on='order_id',
    how='left'
)

In [28]:
# Merge with products on 'product_id'
df_merged = pd.merge(
    df_merged,
    products[['product_id', 'product_category_name']],
    on='product_id',
    how='left'
)

In [29]:
print("✅ Merge Complete!")
print("Total Rows:", len(df_merged))
print("\nFirst 3 rows:")
print(df_merged[['order_id', 'product_id', 'product_category_name', 'price', 'order_status', 'order_purchase_timestamp']].head(3))

✅ Merge Complete!
Total Rows: 112650

First 3 rows:
                           order_id                        product_id  \
0  00010242fe8c5a6d1ba2dd792cb16214  4244733e06e7ecb4970a6e2683c13e61   
1  00018f77f2f0320c557190d7a144bdd3  e5f2d52b802189ee658865ca93d83a8f   
2  000229ec398224ef6ca0657da4fc703e  c777355d18b72b67abbeef9df44fd0fd   

  product_category_name  price order_status order_purchase_timestamp  
0            cool_stuff   58.9    delivered      2017-09-13 08:59:02  
1              pet_shop  239.9    delivered      2017-04-26 10:53:06  
2      moveis_decoracao  199.0    delivered      2018-01-14 14:33:31  


In [30]:
# Create the Total Revenue column
df_merged['Total_Revenue'] = df_merged['price'] + df_merged['freight_value']

In [31]:
# Convert purchase timestamp to datetime
df_merged['order_purchase_timestamp'] = pd.to_datetime(df_merged['order_purchase_timestamp'])

In [32]:
# Keep only delivered, shipped, or invoiced orders
valid_statuses = ['delivered', 'shipped', 'invoiced', 'processing']
df_clean = df_merged[df_merged['order_status'].isin(valid_statuses)].copy()

In [33]:
# Create a Month column (e.g., '2017-09')
df_clean['Month'] = df_clean['order_purchase_timestamp'].dt.to_period('M')

In [34]:
print("✅ Data Cleaned!")
print("Original Rows:", len(df_merged))
print("Cleaned Rows:", len(df_clean))
print("Removed:", len(df_merged) - len(df_clean), "bad orders.")


✅ Data Cleaned!
Original Rows: 112650
Cleaned Rows: 112098
Removed: 552 bad orders.


In [35]:
# Find Top 10 Categories by Revenue
category_revenue = df_clean.groupby('product_category_name')['Total_Revenue'].sum().sort_values(ascending=False).head(10)
print("Top 10 Categories by Revenue:")
print(category_revenue)

Top 10 Categories by Revenue:
product_category_name
beleza_saude              1437665.78
relogios_presentes        1298292.47
cama_mesa_banho           1240386.13
esporte_lazer             1147244.63
informatica_acessorios    1050879.96
moveis_decoracao           899626.04
utilidades_domesticas      771855.68
cool_stuff                 704086.24
automotivo                 678606.64
ferramentas_jardim         579525.20
Name: Total_Revenue, dtype: float64


In [36]:
# Calculate Monthly Revenue
monthly_revenue = df_clean.groupby('Month')['Total_Revenue'].sum()
print("\nMonthly Revenue Trend:")
print(monthly_revenue)


Monthly Revenue Trend:
Month
2016-09        279.69
2016-10      51354.52
2016-12         19.62
2017-01     136943.46
2017-02     283500.07
2017-03     425617.96
2017-04     405669.15
2017-05     582710.83
2017-06     499652.24
2017-07     578753.73
2017-08     661903.52
2017-09     717102.72
2017-10     764756.03
2017-11    1172191.68
2017-12     861526.77
2018-01    1101920.01
2018-02     979486.16
2018-03    1152656.99
2018-04    1156248.89
2018-05    1145686.46
2018-06    1020381.90
2018-07    1039783.58
2018-08     996973.51
2018-09        166.46
Freq: M, Name: Total_Revenue, dtype: float64


In [37]:
# Save the summaries to CSV
category_revenue.to_csv('top_categories_revenue.csv')
monthly_revenue.to_csv('monthly_revenue_trend.csv')

print("✅ Summaries Saved!")

✅ Summaries Saved!


In [38]:
import plotly.express as px

In [39]:
# Reset index for the chart and convert Month to string
monthly_df = monthly_revenue.reset_index()
monthly_df['Month'] = monthly_df['Month'].astype(str)

print("Category Data:")
print(category_revenue)
print("\nMonthly Data:")
print(monthly_df)

Category Data:
product_category_name
beleza_saude              1437665.78
relogios_presentes        1298292.47
cama_mesa_banho           1240386.13
esporte_lazer             1147244.63
informatica_acessorios    1050879.96
moveis_decoracao           899626.04
utilidades_domesticas      771855.68
cool_stuff                 704086.24
automotivo                 678606.64
ferramentas_jardim         579525.20
Name: Total_Revenue, dtype: float64

Monthly Data:
      Month  Total_Revenue
0   2016-09         279.69
1   2016-10       51354.52
2   2016-12          19.62
3   2017-01      136943.46
4   2017-02      283500.07
5   2017-03      425617.96
6   2017-04      405669.15
7   2017-05      582710.83
8   2017-06      499652.24
9   2017-07      578753.73
10  2017-08      661903.52
11  2017-09      717102.72
12  2017-10      764756.03
13  2017-11     1172191.68
14  2017-12      861526.77
15  2018-01     1101920.01
16  2018-02      979486.16
17  2018-03     1152656.99
18  2018-04     1156248.89
19

In [40]:
fig_cat = px.bar(
    x=category_revenue.values,
    y=category_revenue.index,
    orientation='h',
    title='Top 10 Product Categories by Total Revenue',
    labels={'x': 'Total Revenue (BRL)', 'y': 'Product Category'},
    color=category_revenue.values,
    color_continuous_scale='Viridis'
)

# Sort so the highest revenue is at the top
fig_cat.update_layout(yaxis={'categoryorder':'total ascending'})

fig_cat.show()

In [41]:
fig_month = px.line(
    monthly_df,
    x='Month',
    y='Total_Revenue',
    title='Monthly Revenue Trend',
    labels={'Month': 'Month', 'Total_Revenue': 'Total Revenue (BRL)'},
    markers=True
)

fig_month.show()

In [42]:
import os

# 1. Drop the helper columns used for the charts
df_clean = df_clean.drop(columns=['Employment_split', 'Lang_split', 'SatGroup'], errors='ignore')

# 2. Save the final cleaned dataset to CSV
df_clean.to_csv('olist_cleaned_data.csv', index=False)

# 3. Prove the file was actually created
if os.path.exists('olist_cleaned_data.csv'):
    size_mb = os.path.getsize('olist_cleaned_data.csv') / (1024 * 1024)
    print(f"✅ SUCCESS: 'olist_cleaned_data.csv' saved ({size_mb:.2f} MB)")
else:
    print("❌ ERROR: File was not saved.")

✅ SUCCESS: 'olist_cleaned_data.csv' saved (20.82 MB)


In [45]:
from google.colab import files
import os

print("Downloading files...")

# 1. Download the interactive Dashboard
if os.path.exists('ecommerce_dashboard.html'):
    files.download('ecommerce_dashboard.html')
    print("✅ Downloading ecommerce_dashboard.html")
else:
    print("❌ ecommerce_dashboard.html is missing!")

# 2. Download the cleaned dataset
if os.path.exists('olist_cleaned_data.csv'):
    files.download('olist_cleaned_data.csv')
    print("✅ Downloading olist_cleaned_data.csv")
else:
    print("❌ olist_cleaned_data.csv is missing!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloading ecommerce_dashboard.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloading olist_cleaned_data.csv


In [44]:
from IPython.display import display, HTML

# Save the dashboard as an interactive HTML file
fig_dashboard.write_html("ecommerce_dashboard.html")

print("✅ Dashboard generated!")
print("👉 To save this for your presentation, simply right-click the chart below and select 'Save image as...' (or take a screenshot).\n")

# Display the dashboard directly in the notebook
fig_dashboard.show()

✅ Dashboard generated!
👉 To save this for your presentation, simply right-click the chart below and select 'Save image as...' (or take a screenshot).

